# 🏬 PROFITARA — Retail Business Intelligence Platform
### Advanced Analytics & Predictive Intelligence | Built by Dhruv Malhotra & Dev Garg
---
**Stack:** Python · Pandas · XGBoost · Prophet · Scikit-learn · Plotly · Streamlit-ready

**Dataset:** Superstore Sales Dataset (or upload your own CSV)

**Modules:** 25 Analytics Modules covering Sales, Profit, Forecasting, Customer Segmentation, Anomaly Detection & More

In [ ]:
# ── CELL 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q prophet xgboost shap plotly scikit-learn openpyxl
print('✅ All dependencies installed!')

In [ ]:
# ── CELL 2: Imports & Config ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Color Palette — Dark Luxury
GOLD   = '#C9A84C'
DARK   = '#0D0D0D'
CREAM  = '#F5F0E8'
RED    = '#D64045'
TEAL   = '#4ECDC4'
PURPLE = '#7B5EA7'

plt.rcParams.update({
    'figure.facecolor': '#0D0D0D',
    'axes.facecolor':   '#1A1A1A',
    'axes.edgecolor':   GOLD,
    'text.color':       CREAM,
    'axes.labelcolor':  CREAM,
    'xtick.color':      CREAM,
    'ytick.color':      CREAM,
    'grid.color':       '#2A2A2A',
    'grid.linestyle':   '--',
    'font.family':      'monospace'
})

print('✅ Config loaded — Dark Luxury theme active')

In [ ]:
# ── CELL 3: Load Dataset ───────────────────────────────────────────────────────
# Option A: Upload your own CSV
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Option B: Download Superstore dataset automatically
import io, requests

url = 'https://raw.githubusercontent.com/dsaks/superstore/master/Sample%20-%20Superstore.csv'
try:
    r = requests.get(url, timeout=10)
    df = pd.read_csv(io.StringIO(r.text), encoding='latin1')
    print(f'✅ Dataset loaded from URL: {df.shape[0]} rows × {df.shape[1]} cols')
except:
    # Fallback: generate synthetic Superstore-like data
    np.random.seed(42)
    n = 9994
    cats = ['Furniture', 'Office Supplies', 'Technology']
    sub = {'Furniture': ['Chairs','Tables','Bookcases','Furnishings'],
           'Office Supplies': ['Labels','Storage','Art','Binders','Appliances','Paper','Envelopes','Fasteners','Supplies'],
           'Technology': ['Phones','Accessories','Machines','Copiers']}
    regions = ['West','East','Central','South']
    segments = ['Consumer','Corporate','Home Office']
    dates = pd.date_range('2020-01-01', '2023-12-31', periods=n)
    cat_col = np.random.choice(cats, n)
    df = pd.DataFrame({
        'Order Date': dates,
        'Ship Date': dates + pd.to_timedelta(np.random.randint(1,8,n), unit='d'),
        'Category': cat_col,
        'Sub-Category': [np.random.choice(sub[c]) for c in cat_col],
        'Region': np.random.choice(regions, n),
        'Segment': np.random.choice(segments, n),
        'State': np.random.choice(['California','New York','Texas','Washington','Florida','Ohio','Pennsylvania'], n),
        'Sales': np.random.exponential(250, n).round(2),
        'Quantity': np.random.randint(1, 15, n),
        'Discount': np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], n),
        'Profit': np.random.normal(35, 80, n).round(2),
        'Customer ID': ['CUST-' + str(np.random.randint(1000,9999)) for _ in range(n)],
        'Customer Name': ['Customer ' + str(i) for i in np.random.randint(1,500,n)],
        'Product Name': ['Product ' + str(i) for i in np.random.randint(1,1000,n)],
        'Ship Mode': np.random.choice(['Standard Class','Second Class','First Class','Same Day'], n)
    })
    print(f'✅ Synthetic Superstore dataset generated: {n} rows')

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  errors='coerce')
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Quarter']    = df['Order Date'].dt.quarter
df['YearMonth']  = df['Order Date'].dt.to_period('M')
df['DaysToShip'] = (df['Ship Date'] - df['Order Date']).dt.days
df['ProfitMargin'] = (df['Profit'] / df['Sales'].replace(0, np.nan) * 100).round(2)

print(f'Columns: {list(df.columns)}')
df.head(3)

## 📊 MODULE 1–5: Core KPI Overview

In [ ]:
# ── MODULE 1: Executive KPI Summary ───────────────────────────────────────────
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_orders  = df['Order Date'].count()
avg_margin    = df['ProfitMargin'].mean()
total_qty     = df['Quantity'].sum()
unique_custs  = df['Customer ID'].nunique() if 'Customer ID' in df.columns else 'N/A'

print('=' * 55)
print('         🏬  PROFITARA — KPI DASHBOARD')
print('=' * 55)
print(f'  💰  Total Revenue      : ${total_sales:>12,.2f}')
print(f'  📈  Total Profit       : ${total_profit:>12,.2f}')
print(f'  🧾  Total Orders       : {total_orders:>13,}')
print(f'  📦  Units Sold         : {total_qty:>13,}')
print(f'  📉  Avg Profit Margin  : {avg_margin:>12.2f}%')
print(f'  👥  Unique Customers   : {str(unique_custs):>13}')
print('=' * 55)

In [ ]:
# ── MODULE 2: Sales by Category ──────────────────────────────────────────────
cat_sales = df.groupby('Category')[['Sales','Profit']].sum().reset_index().sort_values('Sales', ascending=False)
fig = px.bar(cat_sales, x='Category', y=['Sales','Profit'], barmode='group',
             title='MODULE 2 | Sales & Profit by Category',
             color_discrete_sequence=[GOLD, TEAL],
             template='plotly_dark')
fig.update_layout(title_font_size=16, font_color=CREAM, paper_bgcolor=DARK, plot_bgcolor='#1A1A1A')
fig.show()
cat_sales

In [ ]:
# ── MODULE 3: Monthly Revenue Trend ───────────────────────────────────────────
monthly = df.groupby('YearMonth')['Sales'].sum().reset_index()
monthly['YearMonth'] = monthly['YearMonth'].astype(str)
fig = px.line(monthly, x='YearMonth', y='Sales',
              title='MODULE 3 | Monthly Revenue Trend',
              markers=True, template='plotly_dark',
              color_discrete_sequence=[GOLD])
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.update_traces(line_width=2.5)
fig.show()

In [ ]:
# ── MODULE 4: Regional Performance ────────────────────────────────────────────
region = df.groupby('Region')[['Sales','Profit','Quantity']].sum().reset_index()
region['Margin%'] = (region['Profit']/region['Sales']*100).round(2)
fig = px.bar(region, x='Region', y='Sales', color='Margin%',
             title='MODULE 4 | Regional Performance (Color = Margin%)',
             color_continuous_scale=['#D64045','#C9A84C','#4ECDC4'],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
region

In [ ]:
# ── MODULE 5: Segment Analysis ────────────────────────────────────────────────
seg = df.groupby('Segment')[['Sales','Profit']].sum().reset_index()
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'},{'type':'pie'}]])
fig.add_trace(go.Pie(labels=seg['Segment'], values=seg['Sales'], name='Sales',
                     marker_colors=[GOLD,TEAL,PURPLE]), row=1, col=1)
fig.add_trace(go.Pie(labels=seg['Segment'], values=seg['Profit'], name='Profit',
                     marker_colors=[GOLD,TEAL,PURPLE]), row=1, col=2)
fig.update_layout(title='MODULE 5 | Customer Segment Split — Sales vs Profit',
                  template='plotly_dark', paper_bgcolor=DARK, font_color=CREAM)
fig.show()

## 📦 MODULE 6–10: Product & Sub-Category Deep Dive

In [ ]:
# ── MODULE 6: Top 10 Products by Sales ───────────────────────────────────────
top_products = df.groupby('Product Name')['Sales'].sum().nlargest(10).reset_index()
fig = px.bar(top_products, x='Sales', y='Product Name', orientation='h',
             title='MODULE 6 | Top 10 Products by Revenue',
             color='Sales', color_continuous_scale=[PURPLE, GOLD],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM, yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# ── MODULE 7: Sub-Category Profitability Matrix ────────────────────────────────
subcat = df.groupby('Sub-Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Sales','count')).reset_index()
subcat['Margin%'] = (subcat['Profit']/subcat['Sales']*100).round(2)
fig = px.scatter(subcat, x='Sales', y='Profit', size='Orders', color='Margin%',
                 text='Sub-Category',
                 title='MODULE 7 | Sub-Category Profitability Matrix',
                 color_continuous_scale=[RED, GOLD, TEAL],
                 template='plotly_dark')
fig.update_traces(textposition='top center')
fig.add_hline(y=0, line_dash='dash', line_color=RED, annotation_text='Break-even')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()

In [ ]:
# ── MODULE 8: Discount vs Profit Analysis ────────────────────────────────────
disc = df.groupby('Discount').agg(AvgProfit=('Profit','mean'), Orders=('Sales','count')).reset_index()
fig = px.bar(disc, x='Discount', y='AvgProfit', color='AvgProfit',
             title='MODULE 8 | Discount Rate vs Average Profit',
             color_continuous_scale=[RED, CREAM, TEAL],
             template='plotly_dark')
fig.add_hline(y=0, line_color=RED, line_dash='dot')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
print('🔍 High discounts (>0.3) typically DESTROY margins — verify in your data!')

In [ ]:
# ── MODULE 9: Quantity Distribution by Category ───────────────────────────────
fig = px.box(df, x='Category', y='Quantity', color='Category',
             title='MODULE 9 | Quantity Distribution by Category',
             color_discrete_sequence=[GOLD, TEAL, PURPLE],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()

In [ ]:
# ── MODULE 10: Ship Mode Performance ─────────────────────────────────────────
ship = df.groupby('Ship Mode').agg(
    Orders=('Sales','count'), Revenue=('Sales','sum'),
    AvgDaysShip=('DaysToShip','mean'), AvgProfit=('Profit','mean')
).reset_index().round(2)
fig = px.bar(ship, x='Ship Mode', y='Revenue', color='AvgDaysShip',
             title='MODULE 10 | Shipping Mode — Revenue vs Avg Days to Ship',
             color_continuous_scale=[TEAL, GOLD, RED],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
ship

## 🧠 MODULE 11–15: Customer Intelligence

In [ ]:
# ── MODULE 11: Top 10 Customers by Revenue ───────────────────────────────────
top_custs = df.groupby('Customer Name').agg(
    Revenue=('Sales','sum'), Profit=('Profit','sum'), Orders=('Sales','count')
).nlargest(10, 'Revenue').reset_index()
fig = px.bar(top_custs, x='Revenue', y='Customer Name', orientation='h',
             color='Profit', color_continuous_scale=[RED, GOLD, TEAL],
             title='MODULE 11 | Top 10 Customers by Revenue',
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM,
                  yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# ── MODULE 12: RFM Segmentation ───────────────────────────────────────────────
snapshot = df['Order Date'].max()
rfm = df.groupby('Customer ID').agg(
    Recency  =('Order Date', lambda x: (snapshot - x.max()).days),
    Frequency=('Order Date', 'count'),
    Monetary =('Sales', 'sum')
).reset_index()

rfm['R_Score'] = pd.qcut(rfm['Recency'],   4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal'
    elif score >= 6: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)
seg_counts = rfm['Segment'].value_counts().reset_index()
seg_counts.columns = ['Segment','Count']

fig = px.pie(seg_counts, names='Segment', values='Count',
             title='MODULE 12 | RFM Customer Segmentation',
             color_discrete_sequence=[GOLD, TEAL, PURPLE, RED, CREAM],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, font_color=CREAM)
fig.show()
rfm.head()

In [ ]:
# ── MODULE 13: Customer Lifetime Value (CLV) ──────────────────────────────────
clv = df.groupby('Customer ID').agg(
    TotalSpend=('Sales','sum'), TotalProfit=('Profit','sum'),
    OrderCount=('Sales','count'), AvgOrderValue=('Sales','mean')
).reset_index()
clv['CLV_Score'] = clv['TotalSpend'] * 0.6 + clv['TotalProfit'] * 0.3 + clv['OrderCount'] * 10
top_clv = clv.nlargest(20, 'CLV_Score')

fig = px.scatter(clv, x='OrderCount', y='TotalSpend', size='CLV_Score',
                 color='TotalProfit', color_continuous_scale=[RED, GOLD, TEAL],
                 title='MODULE 13 | Customer Lifetime Value Map',
                 template='plotly_dark', opacity=0.7)
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
print(f'Average CLV Score: {clv["CLV_Score"].mean():.2f}')

In [ ]:
# ── MODULE 14: Geographic Sales Heatmap ───────────────────────────────────────
if 'State' in df.columns:
    state_sales = df.groupby('State')['Sales'].sum().reset_index()
    state_abbrev = {
        'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
        'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
        'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS',
        'Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA',
        'Michigan':'MI','Minnesota':'MN','Mississippi':'MS','Missouri':'MO','Montana':'MT',
        'Nebraska':'NE','Nevada':'NV','New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM',
        'New York':'NY','North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK',
        'Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
        'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT',
        'Virginia':'VA','Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY'
    }
    state_sales['Code'] = state_sales['State'].map(state_abbrev)
    state_sales = state_sales.dropna(subset=['Code'])
    fig = px.choropleth(state_sales, locations='Code', locationmode='USA-states',
                        color='Sales', scope='usa',
                        title='MODULE 14 | US Sales Heatmap by State',
                        color_continuous_scale=[DARK, PURPLE, GOLD],
                        template='plotly_dark')
    fig.update_layout(paper_bgcolor=DARK, font_color=CREAM)
    fig.show()
else:
    print('No State column found — skipping geo heatmap')

In [ ]:
# ── MODULE 15: Repeat vs One-Time Customer Split ──────────────────────────────
order_freq = df.groupby('Customer ID')['Order Date'].count().reset_index()
order_freq.columns = ['Customer ID','OrderCount']
order_freq['Type'] = order_freq['OrderCount'].apply(lambda x: 'Repeat' if x > 1 else 'One-Time')
split = order_freq['Type'].value_counts().reset_index()
split.columns = ['Type','Count']
fig = px.pie(split, names='Type', values='Count',
             title='MODULE 15 | Repeat vs One-Time Customers',
             color_discrete_sequence=[GOLD, RED],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, font_color=CREAM)
fig.show()
print(f'Repeat Customer Rate: {split[split["Type"]=="Repeat"]["Count"].values[0]/split["Count"].sum()*100:.1f}%')

## 🔮 MODULE 16–20: Predictive Analytics

In [ ]:
# ── MODULE 16: Prophet Sales Forecasting (90-Day) ────────────────────────────
from prophet import Prophet

daily_sales = df.groupby('Order Date')['Sales'].sum().reset_index()
daily_sales.columns = ['ds','y']
daily_sales = daily_sales.dropna()

m = Prophet(yearly_seasonality=True, weekly_seasonality=True,
            daily_seasonality=False, changepoint_prior_scale=0.1)
m.fit(daily_sales)
future = m.make_future_dataframe(periods=90)
forecast = m.predict(future)

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_sales['ds'], y=daily_sales['y'],
                          mode='lines', name='Actual', line=dict(color=TEAL, width=1.5)))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'],
                          mode='lines', name='Forecast', line=dict(color=GOLD, width=2)))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'],
                          mode='lines', name='Upper CI',
                          line=dict(color=GOLD, width=0), showlegend=False))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'],
                          mode='lines', name='Lower CI', fill='tonexty',
                          fillcolor='rgba(201,168,76,0.15)',
                          line=dict(color=GOLD, width=0)))
fig.update_layout(title='MODULE 16 | Prophet 90-Day Sales Forecast',
                  template='plotly_dark', paper_bgcolor=DARK,
                  plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()

next_30 = forecast[forecast['ds'] > daily_sales['ds'].max()].head(30)['yhat'].sum()
print(f'📈 Projected Revenue (Next 30 Days): ${next_30:,.2f}')

In [ ]:
# ── MODULE 17: XGBoost Profit Predictor ────────────────────────────────────────
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

feat_cols = ['Sales','Quantity','Discount','Year','Month','Quarter','DaysToShip']
cat_feats  = ['Category','Sub-Category','Region','Segment','Ship Mode']

df_ml = df[feat_cols + cat_feats + ['Profit']].dropna()
le = LabelEncoder()
for col in cat_feats:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

X = df_ml.drop('Profit', axis=1)
y = df_ml['Profit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
r2  = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(f'✅ XGBoost Profit Predictor Trained')
print(f'   R² Score : {r2:.4f}')
print(f'   MAE      : ${mae:.2f}')

In [ ]:
# ── MODULE 18: SHAP Feature Importance ────────────────────────────────────────
import shap

explainer = shap.Explainer(model, X_train)
shap_vals = explainer(X_test[:500])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test[:500], plot_type='bar', show=False,
                  color=GOLD)
plt.title('MODULE 18 | SHAP Feature Importance — Profit Prediction', color=CREAM, fontsize=14)
plt.tight_layout()
plt.show()
print('Features ranked by SHAP impact on profit prediction')

In [ ]:
# ── MODULE 19: Actual vs Predicted Profit Plot ────────────────────────────────
sample = 200
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(sample)), y=y_test.values[:sample],
                          mode='lines', name='Actual Profit', line=dict(color=TEAL)))
fig.add_trace(go.Scatter(x=list(range(sample)), y=preds[:sample],
                          mode='lines', name='Predicted Profit', line=dict(color=GOLD, dash='dash')))
fig.update_layout(title='MODULE 19 | XGBoost: Actual vs Predicted Profit',
                  template='plotly_dark', paper_bgcolor=DARK,
                  plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()

In [ ]:
# ── MODULE 20: Anomaly Detection (Profit Outliers) ────────────────────────────
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)
df_ml2 = df[['Sales','Profit','Quantity','Discount']].dropna()
df_ml2 = df_ml2.copy()
df_ml2['Anomaly'] = iso.fit_predict(df_ml2)
df_ml2['Type'] = df_ml2['Anomaly'].map({1:'Normal', -1:'Anomaly'})

fig = px.scatter(df_ml2, x='Sales', y='Profit', color='Type',
                 color_discrete_map={'Normal': TEAL, 'Anomaly': RED},
                 title='MODULE 20 | Anomaly Detection — Profit Outliers',
                 template='plotly_dark', opacity=0.7)
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
print(f'🚨 Anomalies Detected: {(df_ml2["Type"]=="Anomaly").sum()} out of {len(df_ml2)}')

## 📅 MODULE 21–25: Advanced & Seasonal Intelligence

In [ ]:
# ── MODULE 21: Year-over-Year Growth ─────────────────────────────────────────
yoy = df.groupby('Year')['Sales'].sum().reset_index()
yoy['Growth%'] = yoy['Sales'].pct_change() * 100
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=yoy['Year'], y=yoy['Sales'], name='Revenue',
                      marker_color=GOLD), secondary_y=False)
fig.add_trace(go.Scatter(x=yoy['Year'], y=yoy['Growth%'], name='YoY Growth%',
                          mode='lines+markers', line=dict(color=TEAL, width=2.5)),
              secondary_y=True)
fig.update_layout(title='MODULE 21 | Year-over-Year Revenue Growth',
                  template='plotly_dark', paper_bgcolor=DARK,
                  plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()
yoy

In [ ]:
# ── MODULE 22: Quarterly Revenue Heatmap ──────────────────────────────────────
qheat = df.groupby(['Year','Quarter'])['Sales'].sum().reset_index()
qpivot = qheat.pivot(index='Year', columns='Quarter', values='Sales')

plt.figure(figsize=(9, 5))
sns.heatmap(qpivot, annot=True, fmt='.0f', cmap='YlOrBr',
            linewidths=0.5, linecolor='#2A2A2A',
            annot_kws={'color': 'white', 'size': 10})
plt.title('MODULE 22 | Quarterly Revenue Heatmap', color=CREAM, fontsize=14, pad=15)
plt.xlabel('Quarter', color=CREAM)
plt.ylabel('Year', color=CREAM)
plt.tight_layout()
plt.show()

In [ ]:
# ── MODULE 23: Product Return Risk Scoring ────────────────────────────────────
# Proxy: products with very high discounts + low profit = return risk
risk = df.groupby('Sub-Category').agg(
    AvgDiscount=('Discount','mean'),
    AvgProfit=('Profit','mean'),
    Orders=('Sales','count')
).reset_index()
risk['RiskScore'] = (risk['AvgDiscount'] * 100) - risk['AvgProfit'].clip(upper=0).abs()
risk = risk.sort_values('RiskScore', ascending=False)

fig = px.bar(risk, x='Sub-Category', y='RiskScore', color='RiskScore',
             title='MODULE 23 | Product Return Risk Score by Sub-Category',
             color_continuous_scale=[TEAL, GOLD, RED],
             template='plotly_dark')
fig.update_layout(paper_bgcolor=DARK, plot_bgcolor='#1A1A1A',
                  font_color=CREAM, xaxis_tickangle=-30)
fig.show()

In [ ]:
# ── MODULE 24: Category Revenue Contribution Waterfall ────────────────────────
cat_contrib = df.groupby('Category')['Sales'].sum().reset_index()
cat_contrib['Cumulative'] = cat_contrib['Sales'].cumsum()

fig = go.Figure(go.Waterfall(
    name='Revenue',
    orientation='v',
    x=cat_contrib['Category'].tolist() + ['Total'],
    y=cat_contrib['Sales'].tolist() + [cat_contrib['Sales'].sum()],
    measure=['relative'] * len(cat_contrib) + ['total'],
    connector={'line': {'color': GOLD}},
    decreasing={'marker': {'color': RED}},
    increasing={'marker': {'color': TEAL}},
    totals={'marker': {'color': GOLD}}
))
fig.update_layout(title='MODULE 24 | Revenue Contribution Waterfall',
                  template='plotly_dark', paper_bgcolor=DARK,
                  plot_bgcolor='#1A1A1A', font_color=CREAM)
fig.show()

In [ ]:
# ── MODULE 25: Executive Summary Report ───────────────────────────────────────
best_cat   = df.groupby('Category')['Profit'].sum().idxmax()
best_region = df.groupby('Region')['Sales'].sum().idxmax()
worst_disc  = df.groupby('Discount')['Profit'].mean().idxmin()
peak_month  = df.groupby('Month')['Sales'].sum().idxmax()
months_name = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('=' * 60)
print('       📋  PROFITARA — EXECUTIVE INTELLIGENCE REPORT')
print('=' * 60)
print(f'  💰  Total Revenue        : ${df["Sales"].sum():>12,.2f}')
print(f'  📈  Total Profit         : ${df["Profit"].sum():>12,.2f}')
print(f'  📉  Avg Profit Margin    : {df["ProfitMargin"].mean():>11.2f}%')
print(f'  🏆  Best Category        : {best_cat}')
print(f'  🌍  Top Region           : {best_region}')
print(f'  📅  Peak Sales Month     : {months_name[peak_month]}')
print(f'  ⚠️   Most Harmful Discount: {worst_disc*100:.0f}%')
print(f'  🤖  XGBoost R² Score     : {r2:.4f}')
print('=' * 60)
print()
print('✅ All 25 Modules Complete — PROFITARA v2.0')
print('   Built by: Dhruv Malhotra & Dev Garg')
print('   Stack: Python · Pandas · XGBoost · Prophet · Plotly')